In [ ]:
import re
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from scrapling.fetchers import StealthyFetcher

BASE = "https://www.bloomberg.com"
LATEST = f"{BASE}/latest"


def fetch(url):
    """StealthyFetcher drives sync playwright -> keep it off the notebook's event loop."""
    return StealthyFetcher.fetch(url, headless=True, network_idle=True, google_search=False)


def in_thread(fn, *args):
    with ThreadPoolExecutor(1) as pool:
        return pool.submit(fn, *args).result()


def parse_latest(page):
    rows = []
    for s in page.css('.Latest_storyPadding__GBJUE'):
        a = s.css('a.Latest_storyLink__80QVD')
        if not a:
            continue
        spans, times = a[0].css('span'), s.css('time')
        rows.append({
            'headline': spans[0].text if spans else None,
            'url': BASE + a[0].attrib.get('href', ''),
            'timestamp': times[0].text if times else None,
        })
    return rows


def extract_row(url):
    page = fetch(url)
    meta = page.css('meta[property="og:title"], meta[name="og:title"]')
    normalized = re.sub(r'[\s_-]+', '', page.html_content.lower())
    return {
        'url': url,
        'title': meta[0].attrib.get('content') if meta else None,
        'is_bloomberg_exclusive': 'bloombergexclusives' in normalized,
    }

In [ ]:
df = pd.DataFrame(parse_latest(in_thread(fetch, LATEST)))
print(f"{len(df)} stories found")
df.head(3)

In [ ]:
import os

CSV_PATH = 'scraped.csv'
seen_urls = set(pd.read_csv(CSV_PATH)['url']) if os.path.exists(CSV_PATH) else set()

new_urls = df.loc[~df['url'].isin(seen_urls), 'url']
print(f"{len(new_urls)} new of {len(df)} stories")

with ThreadPoolExecutor(4) as pool:
    scraped = pd.DataFrame(list(pool.map(extract_row, new_urls)))

if not scraped.empty:
    scraped.to_csv(CSV_PATH, mode='a', header=not os.path.exists(CSV_PATH), index=False, encoding='utf-8-sig')

scraped

,url,title,is_bloomberg_exclusive
0,https://www.bloomberg.com/opinion/articles/202...,We’re Going About Nuclear Shipping the Wrong Way,False
1,https://www.bloomberg.com/news/articles/2026-0...,"Elliott Builds Deutsche Telekom Stake, Opposes...",True
2,https://www.bloomberg.com/news/articles/2026-0...,Estonian Defense Minister Pevkur Resigns Over ...,False


In [27]:
import re, asyncio
from datetime import datetime
from scrapling.fetchers import AsyncStealthySession

LATEST, seen, rows = "https://www.bloomberg.com/latest", set(), []
log = lambda m: print(f"[{datetime.now():%H:%M:%S}] {m}", flush=True)

def parse(page):
    out = []
    for s in page.css('.Latest_storyPadding__GBJUE'):
        a, t = s.css('a.Latest_storyLink__80QVD'), s.css('time')
        if a:
            spans = a[0].css('span')
            out.append({'headline': spans[0].text if spans else None,
                        'url': 'https://www.bloomberg.com' + a[0].attrib.get('href', ''),
                        'timestamp': t[0].text if t else None})
    return out

async def check(sess, r):
    try:
        p = await sess.fetch(r['url'], network_idle=True)
        r['is_bloomberg_exclusive'] = 'bloombergexclusives' in re.sub(r'[\s_-]+', '', p.html_content.lower())
        log(f"    {'EXCLUSIVE' if r['is_bloomberg_exclusive'] else 'not excl.'} [{p.status}] {r['headline']}")
    except Exception as e:
        r['is_bloomberg_exclusive'] = None
        log(f"    CHECK FAILED {r['url']} -> {type(e).__name__}: {e}")
    return r

async def monitor(every=60):
    log("launching browser...")
    async with AsyncStealthySession(max_pages=5, headless=True, network_idle=True) as sess:
        primed = parse(await sess.fetch(LATEST))
        seen.update(r['url'] for r in primed)
        log(f"primed with {len(primed)} existing stories — watching every {every}s")
        cycle = 0
        while True:
            await asyncio.sleep(every)
            cycle += 1
            t0 = asyncio.get_event_loop().time()
            try:
                stories = parse(await sess.fetch(LATEST))
            except Exception as e:
                log(f"#{cycle} refresh FAILED -> {type(e).__name__}: {e}"); continue
            new = [r for r in stories if r['url'] not in seen]
            log(f"#{cycle} refreshed in {asyncio.get_event_loop().time()-t0:.1f}s — {len(stories)} on page, {len(new)} new, {len(seen)} seen")
            if not new:
                continue
            for r in new:
                log(f"  NEW: {r['headline']}  ({r['timestamp']})")
            seen.update(r['url'] for r in new)
            done = await asyncio.gather(*(check(sess, r) for r in new))
            rows.extend(done)
            log(f"#{cycle} done — {sum(1 for r in done if r['is_bloomberg_exclusive'])} exclusive of {len(done)}; {len(rows)} rows total")

await monitor()


[16:16:52] launching browser...


[2026-09-02 16:17:28] INFO: Fetched (200) <GET https://www.bloomberg.com/latest> (referer: https://www.google.com/)


[16:17:28] primed with 25 existing stories — watching every 60s


[2026-09-02 16:18:59] INFO: Fetched (200) <GET https://www.bloomberg.com/latest> (referer: https://www.google.com/)


[16:18:59] #1 refreshed in 31.4s — 25 on page, 0 new, 25 seen


...

[2026-09-02 16:37:34] INFO: Fetched (403) <GET https://www.bloomberg.com/latest> (referer: https://www.google.com/)


[16:37:34] #19 refreshed in 2.1s — 0 on page, 0 new, 25 seen


[2026-09-02 16:38:37] INFO: Fetched (403) <GET https://www.bloomberg.com/latest> (referer: https://www.google.com/)


[16:38:37] #20 refreshed in 2.1s — 0 on page, 0 new, 25 seen


CancelledError: 